# Taxonomic Assignment and Preliminary Taxonomic Quality Assessment

This notebook/script assigns taxonomic classifications to ASVs inferred by DADA2 using the SILVA v138.2 reference database. Taxonomic assignments are evaluated to assess classification success across taxonomic ranks, summarize the overall composition of the dataset, and construct the sample and taxonomic metadata required for downstream diversity, differential abundance, and phylogenetic analyses.

Taxonomic assignments performed based on:  
- DADA2 tutorial: `https://benjjneb.github.io/dada2/assign.html`  
- qiime2r documentation: `https://github.com/jbisanz/qiime2R`  
- SILVA training classifier from here DADA2 Github: `https://benjjneb.github.io/dada2/training.html`  



### Library Import

In [2]:

suppressPackageStartupMessages({

    ########### main libraries ###########
    library(qiime2R) # QIIME 2 import
    library(dada2) # taxonomy assignment
    
    # mia data structures
    library(TreeSummarizedExperiment)
    library(mia)
    library(miaViz) # mia visualizatioon
    

    ########### data manipulation tools ###########
    library(tidyverse)
    
    ########### visualization libraries ###########
    
    # library(ggpubr)
    # library(pheatmap)
    # library(readr)
    # library(RColorBrewer)
    # library(colorspace)
    # library(cowplot)
    # library(grid)
    # library(patchwork)
    # library(gridExtra)
    # library(ggrepel)
    # library(ggpubr)
    # library(ggthemes)
    # library(RColorBrewer)
    # library(patchwork)
    # library(cowplot)
    
    ########### resource management tools ###########
    library(tictoc)

    ### loaded by tidyverse
    # library(ggplot2)
    # library(tidyr)
    # library(dplyr)
    # library(readr)
    # library(stringr)
})



warnLevel <- getOption('warn')
options(warn = -1)



### Arguments Piped in with R script Conversion

In [3]:
args <- commandArgs(trailingOnly = TRUE)
if (length(args) >= 2) {
    PROJECT_NAME <- args[1]
    PROJECT_DIR  <- normalizePath(args[2])
} else {
    # defaults for notebook 
    PROJECT_NAME <- "ERP000133"
    PROJECT_DIR  <- normalizePath("..")
}

### Input File Paths & Read Files

In [4]:
INITIAL_QC_DIR <- file.path(PROJECT_DIR, "initial_qc")
DADA2_INPUT_DIR <- file.path(INITIAL_QC_DIR, "qiime2", "dada2")
PHYLO_INPUT_DIR <- file.path(INITIAL_QC_DIR, "qiime2", "phylo")

SILVA_TRAINSET_GENUS_FILE <- file.path(PROJECT_DIR, "taxonomy_training_classifiers", "silva_nr99_v138.2_toGenus_trainset.fa.gz")
SILVA_TRAINSET_SPECIES_FILE <- file.path(PROJECT_DIR, "taxonomy_training_classifiers", "silva_nr99_v138.2_toSpecies_trainset.fa.gz")


In [5]:
dada2_file_path <- function(suffix) {
    filename = file.path(DADA2_INPUT_DIR, paste0(PROJECT_NAME, ".qiime.dada2.", suffix))
    return (filename)
}

rep_seqs <- read_qza(dada2_file_path("rep_seqs.qza")) # representative sequences, generated by QIIME 2's implementation of DADA2
asv_table <- read_qza(dada2_file_path("asv_table.qza")) # ASV table, generated by QIIME 2's implementation of DADA2
denoising_stats <- read_qza(dada2_file_path("denoising_stats.qza")) # denoising statistics, generated by QIIME 2's implementation of DADA2
base_transition_stats <- read_qza(dada2_file_path("base_transition_stats.qza"))

rooted_tree <- read_qza(file.path(PHYLO_INPUT_DIR, paste0(PROJECT_NAME, ".qiime.phylo.", "rooted_tree.qza"))) # rooted tree of sequences, generated by QIIME 2's phylogeny function

sample_metadata_tse <- read_tsv(
    file.path(PROJECT_DIR, "summary_tables", paste0(PROJECT_NAME, ".qc_summary_table_for_TSE.tsv")), # sample metadata table made for making the TSE object, generated by the create_qc_summary_tables script/notebook
    show_col_types = FALSE
) |>
    tibble::column_to_rownames("sample_id")



### Make Output Directory

In [ ]:
# make file output directory if it doesn't already exist
TAXA_DIR <- file.path(PROJECT_DIR, "taxonomy_assignment")
dir.create(TAXA_DIR, recursive = TRUE, showWarnings = FALSE)
REPORT_TABLE_DIR <- file.path(PROJECT_DIR, "report_tables")
SUMMARY_TABLE_DIR <- file.path(PROJECT_DIR, "summary_tables") # more extensive tables not for the report go here
dir.create(REPORT_TABLE_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(SUMMARY_TABLE_DIR, recursive = TRUE, showWarnings = FALSE)


### Prepare representative sequences for taxonomy assignment

In [7]:
# print(class(rep_seqs$data)) # check class of representative sequences data element
# str(rep_seqs$data) # check object structure

In [8]:
# extract sequences, name by rep_seq data names
seqs <- as.character(rep_seqs$data)
names(seqs) <- names(rep_seqs$data)
# head(seqs)

In [9]:
# print(length(seqs)) # check how many representatice sequences there are in the dataset
# head(seqs)

### Perform genus & species level taxonomy assignment

Here I assign taxonomies using the SILVA training datasets, at both the genus level and species level. I performed species level taxonomy assignment just out of curiosity.

In [10]:
set.seed(100) # initialize random number generator for reproducibility

In [11]:
# Assign taxonomies at the genus level
tic()
taxa_genus <- assignTaxonomy(
    seqs,
    SILVA_TRAINSET_GENUS_FILE,
    multithread = TRUE
)
toc()

73.904 sec elapsed


In [ ]:
# Assign taxonomies at the species level
tic()
taxa_species <- assignTaxonomy(
    seqs,
    SILVA_TRAINSET_SPECIES_FILE,
    multithread = TRUE
)
toc()

##### Look at QC summaries

In [ ]:
taxonomy_summary <- tibble::tibble(
    metric = c(
        "ASVs with no genus assignment",
        "ASVs with no species assignment",
        "Unique assigned genera",
        "Total ASVs"
    ),
    value = c(
        sum(is.na(taxa_genus[, "Genus"])),
        sum(is.na(taxa_species[, "Species"])),
        length(unique(na.omit(taxa_genus[, "Genus"]))),
        nrow(taxa_genus)
    )
)

# add percentages
taxonomy_summary <- taxonomy_summary |>
    dplyr::mutate(
        percent = c(
            100 * value[1] / nrow(taxa_genus),
            100 * value[2] / nrow(taxa_species),
            NA,
            NA
        )
    )

# print a clean taxonomy QC summary
cat("TAXONOMY ASSIGNMENT SUMMARY\n")
cat("===========================\n\n")
print(taxonomy_summary, n = Inf)

# check the top most common genera by number of ASVs assigned to each genus
top_genera <- sort(
    table(taxa_genus[, "Genus"]),
    decreasing = TRUE
) |>
    head(10) |>
    as.data.frame() |>
    dplyr::rename(
        Genus = Var1,
        ASV_count = Freq
    )

cat("\n\nTOP 10 GENERA BY NUMBER OF ASVs\n")
cat("===============================\n\n")
print(top_genera, row.names = FALSE)

cat("\n\nTAXONOMY OBJECT INFO\n")
cat("====================\n\n")

cat("class:      ", paste(class(taxa_genus), collapse = ", "), "\n")
cat("dimensions: ", nrow(taxa_genus), "ASVs x", ncol(taxa_genus), "taxonomic ranks\n")
cat("ranks:      ", paste(colnames(taxa_genus), collapse = ", "), "\n")

cat("\n\nFIRST 6 TAXONOMY ASSIGNMENTS\n")
cat("============================\n\n")

taxa_genus |>
    head(6) |>
    as.data.frame() |>
    tibble::rownames_to_column("ASV_sequence") |>
    print()

In [ ]:
# check how many ASVs could be classified at each taxonomic rank
taxonomic_assignment_summary <- tibble::tibble(
    Taxonomic_rank = colnames(taxa_genus),
    Assigned_ASVs = colSums(!is.na(taxa_genus))
) |>
    dplyr::mutate(
        Total_ASVs = nrow(taxa_genus),
        Percent_assigned = round(
            100 * Assigned_ASVs / Total_ASVs,
            digits = 1
        )
    )

cat("TAXONOMIC ASSIGNMENT BY RANK\n")
cat("============================\n\n")

print(
    taxonomic_assignment_summary,
    n = Inf
)

In [1]:
taxonomic_assignment_summary

ERROR: Error: object 'taxonomic_assignment_summary' not found


In [ ]:
# save table for quarto plot
write_tsv( 
    taxonomic_assignment_summary,
    file = file.path(QUARTO_PLOT_TABLE_DIR, paste0(PROJECT_NAME, ".alpha_diversity_for_plot.tsv")),
    quote = "needed"
)




### Quickly Check Species level assignment

Only a few sequences were successfully labelled with species level taxa assignments, so I have not continued using this object

In [14]:
# make a few basic taxonomy QC summaries
taxonomy_summary_species <- tibble::tibble(
    metric = c(
        "ASVs with no genus assignment",
        "ASVs with no species assignment",
        "Unique assigned genera",
        "Unique assigned species",
        "Total ASVs"
    ),
    value = c(
        sum(is.na(taxa_species[, "Genus"])),
        sum(is.na(taxa_species[, "Species"])),
        length(unique(na.omit(taxa_species[, "Genus"]))),
        length(unique(na.omit(taxa_species[, "Species"]))),
        nrow(taxa_species)
    )
)

# add percentages 
taxonomy_summary_species <- taxonomy_summary_species |>
    dplyr::mutate(
        percent = c(
            100 * value[1] / nrow(taxa_species),
            100 * value[2] / nrow(taxa_species),
            NA,
            NA,
            NA
        )
    )


cat("TAXONOMY ASSIGNMENT SUMMARY\n")
cat("===========================\n\n")
print(taxonomy_summary_species, n = Inf)







# check the top most common genera by number of ASVs assigned to each genus
top_genera_species <- sort(
    table(taxa_species[, "Genus"]),
    decreasing = TRUE
) |>
    head(10) |>
    as.data.frame() |>
    dplyr::rename(
        Genus = Var1,
        ASV_count = Freq
    )

cat("\n\nTOP 10 GENERA BY NUMBER OF ASVs\n")
cat("===============================\n\n")
print(top_genera_species, row.names = FALSE)

# check the top most common species by number of ASVs assigned to each species
top_species_species <- sort(
    table(taxa_species[, "Species"]),
    decreasing = TRUE
) |>
    head(10) |>
    as.data.frame() |>
    dplyr::rename(
        Species = Var1,
        ASV_count = Freq
    )

cat("\n\nTOP 10 SPECIES BY NUMBER OF ASVs\n")
cat("================================\n\n")
print(top_species_species, row.names = FALSE)

cat("\n\nTAXONOMY OBJECT INFO\n")
cat("====================\n\n")

# cat("class:      ", paste(class(taxa_species), collapse = ", "), "\n")
cat("dimensions: ", nrow(taxa_species), "ASVs x", ncol(taxa_species), "taxonomic ranks\n")
cat("ranks:      ", paste(colnames(taxa_species), collapse = ", "), "\n")

cat("\n\nFIRST 6 TAXONOMY ASSIGNMENTS\n")
cat("============================\n\n")

taxa_species |>
    head(6) |>
    as.data.frame() |>
    tibble::rownames_to_column("ASV_sequence") |>
    print()

TAXONOMY ASSIGNMENT SUMMARY

# A tibble: 5 × 3
  metric                          value percent
  <chr>                           <int>   <dbl>
1 ASVs with no genus assignment     349    24.9
2 ASVs with no species assignment  1106    78.9
3 Unique assigned genera            166    NA  
4 Unique assigned species           113    NA  
5 Total ASVs                       1402    NA  


TOP 10 GENERA BY NUMBER OF ASVs

                         Genus ASV_count
                       Blautia        75
              Faecalibacterium        48
                     Roseburia        48
                   Bacteroides        47
 Lachnospiraceae NK4A136 group        28
                       Leyella        24
                     Segatella        24
                     Alistipes        23
 Christensenellaceae R-7 group        23
               Parabacteroides        23


TOP 10 SPECIES BY NUMBER OF ASVs

        Species ASV_count
  inulinivorans        19
    prausnitzii        18
 parainfluenzae  

In [15]:
# check how many ASVs could be classified at each taxonomic rank
taxonomic_assignment_summary_species <- tibble::tibble(
    Taxonomic_rank = colnames(taxa_species),
    Assigned_ASVs = colSums(!is.na(taxa_species))
) |>
    dplyr::mutate(
        Total_ASVs = nrow(taxa_species),
        Percent_assigned = round(
            100 * Assigned_ASVs / Total_ASVs,
            digits = 1
        )
    )

cat("TAXONOMIC ASSIGNMENT BY RANK\n")
cat("============================\n\n")

print(
    taxonomic_assignment_summary_species,
    n = Inf
)

TAXONOMIC ASSIGNMENT BY RANK

# A tibble: 7 × 4
  Taxonomic_rank Assigned_ASVs Total_ASVs Percent_assigned
  <chr>                  <dbl>      <int>            <dbl>
1 Kingdom                 1401       1402             99.9
2 Phylum                  1398       1402             99.7
3 Class                   1396       1402             99.6
4 Order                   1381       1402             98.5
5 Family                  1291       1402             92.1
6 Genus                   1053       1402             75.1
7 Species                  296       1402             21.1


#### Save Tables

In [16]:
# genus
#    rds
saveRDS(
    taxa_genus,
    file = file.path(TAXA_DIR, paste0(PROJECT_NAME, ".taxa_genus.rds"))
)

#    tsv
write_tsv(
    as.data.frame(taxa_genus) |>
        tibble::rownames_to_column("feature_id"),
   file = file.path(TAXA_DIR, paste0(PROJECT_NAME, ".taxa_genus.tsv"))
)




# summary
write_tsv(
   taxonomy_summary,
   file = file.path(TAXA_DIR, paste0(PROJECT_NAME, ".taxa_genus.taxonomy_summary.tsv"))
)


# assignment summary
write_tsv(
   taxonomic_assignment_summary,
   file = file.path(TAXA_DIR, paste0(PROJECT_NAME, ".taxa_genus.taxonomy_assignment_rate_summary.tsv"))
)


In [17]:
# genus
#    rds
saveRDS(
    taxa_species,
    file = file.path(TAXA_DIR, paste0(PROJECT_NAME, ".taxa_species.rds"))
)

#    tsv
write_tsv(
    as.data.frame(taxa_species) |>
        tibble::rownames_to_column("feature_id"),
   file = file.path(TAXA_DIR, paste0(PROJECT_NAME, ".taxa_species.tsv"))
)




# summary
write_tsv(
   taxonomy_summary_species,
   file = file.path(TAXA_DIR, paste0(PROJECT_NAME, ".taxa_species.taxonomy_summary.tsv"))
)


# assignment summary
write_tsv(
   taxonomic_assignment_summary_species,
   file = file.path(TAXA_DIR, paste0(PROJECT_NAME, ".taxa_species.taxonomy_assignment_rate_summary.tsv"))
)


### Create TSE object

In [18]:
head(asv_table$data)

,10BF,10EU,11BF,11EU,12BF,12EU,13BF,13EU,15BF,16BF,⋯,3BF,3EU,4BF,5EU,6BF,6EU,7BF,8BF,8EU,9BF
e5d80e368534833f89790dddf7064358,5421,0,5943,0,12010,0,3966,0,0,4726,⋯,1354,0,675,0,0,0,2334,0,0,0
6415b3db426daec918b055048ef228ff,73,1036,0,5186,45,148,0,1876,28,126,⋯,236,0,152,249,120,574,78,62,469,99
f69c9987c8697d3a60d760d251c88f79,0,0,0,0,0,0,0,0,0,0,⋯,5224,0,4491,0,0,0,0,0,0,0
f966fcbc558ce15a073c7d21f8f68341,72,202,44,0,50,1594,84,0,0,0,⋯,0,285,0,785,87,1556,221,101,2543,0
ced978e18dcdbfae5c48a5042e6022f2,0,0,0,0,0,0,0,0,2564,0,⋯,0,0,0,0,1071,0,0,1247,0,2387
c85a814f944acc30780f2567e058cad9,1692,0,944,0,1624,0,831,0,0,627,⋯,638,0,411,0,0,0,712,0,0,0


In [19]:
# taxa_genus2 <- taxa_genus 

In [20]:
# switch the taxonomy rownames from sequences back to the QIIME ASV ids
rownames(taxa_genus) <- names(rownames(taxa_genus))

# sum(rownames(asv_table$data) %in% rownames(taxa_genus))

In [21]:
# reorder taxa genus table to match ASV
taxa_genus <- taxa_genus[rownames(asv_table$data), , drop = FALSE]


In [22]:

# reorder sample metadata to match ASV
sample_metadata_tse <- sample_metadata_tse[
    colnames(asv_table$data),
    ,
    drop = FALSE
]
# sample_metadata_tse

In [24]:
# dim(asv_table$data)
# dim(taxa_genus)

# head(rownames(asv_table$data))
# head(rownames(taxa_genus))

# sum(rownames(asv_table$data) %in% rownames(taxa_genus))
# length(rownames(asv_table$data))

In [25]:


tse <- TreeSummarizedExperiment(
    assays = list(counts = as.matrix(asv_table$data)),
    rowData = S4Vectors::DataFrame(taxa_genus),
    colData = S4Vectors::DataFrame(sample_metadata_tse)
)

# colData(tse)

#### Inspect the resulting object

In [44]:
head(colData(tse))

DataFrame with 6 rows and 26 columns
           group      run_id population_group age_years         sex
     <character> <character>      <character> <numeric> <character>
10BF          BF   ERR011058               BF         6      female
10EU          EU   ERR011059               EU         5        male
11BF          BF   ERR011060               BF         5        male
11EU          EU   ERR011061               EU         5        male
12BF          BF   ERR011062               BF         6        male
12EU          EU   ERR011063               EU         6        male
     reported_ethnicity     antibiotic_history      delivery_mode
            <character>            <character>        <character>
10BF      black african             none known natural childbirth
10EU          caucasian more than 6 months s.. natural childbirth
11BF      black african             none known natural childbirth
11EU          caucasian more than 6 months s.. natural childbirth
12BF      black african

In [27]:
dim(tse)


[1] 1402   29

In [28]:

assayNames(tse)


[1] "counts"

In [29]:
head(rowData(tse))


DataFrame with 6 rows and 6 columns
                                     Kingdom         Phylum          Class
                                 <character>    <character>    <character>
e5d80e368534833f89790dddf7064358    Bacteria   Bacteroidota    Bacteroidia
6415b3db426daec918b055048ef228ff    Bacteria      Bacillota     Clostridia
f69c9987c8697d3a60d760d251c88f79    Bacteria Actinomycetota Actinobacteria
f966fcbc558ce15a073c7d21f8f68341    Bacteria      Bacillota     Clostridia
ced978e18dcdbfae5c48a5042e6022f2    Bacteria   Bacteroidota    Bacteroidia
c85a814f944acc30780f2567e058cad9    Bacteria   Bacteroidota    Bacteroidia
                                             Order             Family
                                       <character>        <character>
e5d80e368534833f89790dddf7064358     Bacteroidales     Prevotellaceae
6415b3db426daec918b055048ef228ff   Oscillospirales    Ruminococcaceae
f69c9987c8697d3a60d760d251c88f79 Bifidobacteriales Bifidobacteriaceae
f966fcbc558ce1

In [30]:
head(colData(tse))

DataFrame with 6 rows and 24 columns
           group      run_id population_group age_years         sex
     <character> <character>      <character> <numeric> <character>
10BF          BF   ERR011058               BF         6      female
10EU          EU   ERR011059               EU         5        male
11BF          BF   ERR011060               BF         5        male
11EU          EU   ERR011061               EU         5        male
12BF          BF   ERR011062               BF         6        male
12EU          EU   ERR011063               EU         6        male
     reported_ethnicity     antibiotic_history      delivery_mode
            <character>            <character>        <character>
10BF      black african             none known natural childbirth
10EU          caucasian more than 6 months s.. natural childbirth
11BF      black african             none known natural childbirth
11EU          caucasian more than 6 months s.. natural childbirth
12BF      black african

In [31]:
colData(tse)$final_read_depth <- colSums(assay(tse, "counts"))
colData(tse)$observed_ASVs <- colSums(assay(tse, "counts") > 0)

In [32]:
colData(tse)$final_read_depth

10BF  10EU  11BF  11EU  12BF  12EU  13BF  13EU  15BF  16BF  17BF  17EU  18EU 
15404 11523 11873 14050 20394 11530 13728 17599  9982 16975 15216 12008 14805 
 19EU   1EU  20EU  21EU   2BF   2EU   3BF   3EU   4BF   5EU   6BF   6EU   7BF 
11128  7864 16090 18067  8200 11357 12937 12527 12200 20332  9450 15695 10773 
  8BF   8EU   9BF 
10599 16566 10520

In [33]:
colData(tse)$observed_ASVs

10BF 10EU 11BF 11EU 12BF 12EU 13BF 13EU 15BF 16BF 17BF 17EU 18EU 19EU  1EU 20EU 
  90  128   55   56   76   87   96   85   39   59   69  112  102  125   55  179 
21EU  2BF  2EU  3BF  3EU  4BF  5EU  6BF  6EU  7BF  8BF  8EU  9BF 
 134  134   48   50   54   75  127  165  132  156  177  104   82

In [34]:
# Check for missing kingdom assigment
table(rowData(tse)$Kingdom, useNA = "ifany")



Bacteria     <NA> 
    1401        1 

In [35]:
# Check for missing Phylum assigment
table(rowData(tse)$Phylum, useNA = "ifany")



         Actinomycetota               Bacillota            Bacteroidota 
                     45                     962                     295 
       Campylobacterota          Fusobacteriota         Patescibacteria 
                      7                       1                       3 
         Pseudomonadota           Spirochaetota Thermodesulfobacteriota 
                     69                       8                       6 
      Verrucomicrobiota                    <NA> 
                      1                       5 

In [36]:
as.data.frame(rowData(tse))[is.na(rowData(tse)$Kingdom), ]

rownames(tse)[is.na(rowData(tse)$Kingdom)]

,Kingdom,Phylum,Class,Order,Family,Genus
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
b8a4550b039005f68cce790838d232c5,NA,NA,NA,NA,NA,NA


[1] "b8a4550b039005f68cce790838d232c5"

In [37]:
# What's the sequence of the one missing NA Kingdom ASV? 
rep_seqs$data[rownames(tse)[is.na(rowData(tse)$Kingdom)]]

DNAStringSet object of length 1:
    width seq                                               names               
[1]    75 CCCTGAATGATGTACCGGGCCGG...CGGCCTTGAAGCTGCGATACTCC b8a4550b039005f68...

In [38]:
# Check for non-bacterial species 
unique(rowData(tse)$Family[grepl("mitochond", rowData(tse)$Family, ignore.case = TRUE)])
unique(rowData(tse)$Order[grepl("chloroplast", rowData(tse)$Order, ignore.case = TRUE)])

character(0)

character(0)

In [39]:
# the ASV(s) with no kingdom assignment
missing_bac <- tse[is.na(rowData(tse)$Kingdom), ]
as.data.frame(rowData(missing_bac))

,Kingdom,Phylum,Class,Order,Family,Genus
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
b8a4550b039005f68cce790838d232c5,NA,NA,NA,NA,NA,NA


In [40]:
# NA counts across all samples
as.data.frame(assay(tse, "counts")[is.na(rowData(tse)$Kingdom), ])

,"assay(tse, ""counts"")[is.na(rowData(tse)$Kingdom), ]"
,<dbl>
10BF,0
10EU,0
11BF,0
11EU,0
12BF,0
12EU,0
13BF,0
13EU,0
15BF,0


#### Save TSE object

In [43]:
#    rds
saveRDS(
    tse,
    file = file.path(TAXA_DIR, paste0(PROJECT_NAME, ".ASV_with_taxa.TSE.rds"))
)
